# Codebase Q&A — Step 1: Load & Chunk
Experiment notebook. Once logic is solid, it moves to `.py` modules.

**Goal:** Load the `httpx` repo → chunk Python files with AST → inspect chunk quality

## 0. Install dependencies

In [ ]:
# Run once
# !pip install sentence-transformers chromadb langchain anthropic

## 1. Clone the repo (run once)

In [ ]:
import subprocess, os

if not os.path.exists('./httpx'):
    subprocess.run(['git', 'clone', 'https://github.com/encode/httpx.git'], check=True)
    print('✅ Cloned httpx')
else:
    print('✅ httpx already exists')

## 2. Load all files

In [ ]:
import sys
sys.path.append('..')  # so we can import from project root

from ingestion.loader import load_repo, summarize_loaded_files

files = load_repo('./httpx')
summarize_loaded_files(files)

In [ ]:
# Inspect a single file
py_files = [f for f in files if f.language == 'python']
sample_file = py_files[0]

print(f'File     : {sample_file.relative_path}')
print(f'Language : {sample_file.language}')
print(f'Size     : {sample_file.size_bytes} bytes')
print(f'\nFirst 500 chars:\n{"-"*40}')
print(sample_file.content[:500])

## 3. Chunk files (AST for Python)

In [ ]:
from ingestion.chunker import chunk_all_files, chunk_python_file

all_chunks = chunk_all_files(files)

In [ ]:
# Inspect chunk type distribution
from collections import Counter

node_types = Counter(c.node_type for c in all_chunks)
print('Chunk types:')
for k, v in node_types.most_common():
    print(f'  {k:<25} {v:>4}')

In [ ]:
# Look at a function chunk in detail
fn_chunks = [c for c in all_chunks if c.node_type == 'function']
sample = fn_chunks[5]  # change index to explore

print(f'File     : {sample.relative_path}')
print(f'Function : {sample.node_name}')
print(f'Lines    : {sample.start_line}–{sample.end_line}')
print(f'Length   : {len(sample.text)} chars')
print(f'\n{"-"*50}\n{sample.text}')

In [ ]:
# Look at a class chunk
class_chunks = [c for c in all_chunks if c.node_type == 'class']
sample = class_chunks[0]

print(f'File  : {sample.relative_path}')
print(f'Class : {sample.node_name}')
print(f'Lines : {sample.start_line}–{sample.end_line}')
print(f'\n{"-"*50}\n{sample.text[:600]}')

## 4. Quality Check
Check chunk size distribution — ideally most chunks are 200–1500 chars

In [ ]:
import statistics

sizes = [len(c.text) for c in all_chunks]
print(f'Total chunks : {len(sizes)}')
print(f'Min size     : {min(sizes)} chars')
print(f'Max size     : {max(sizes)} chars')
print(f'Mean size    : {statistics.mean(sizes):.0f} chars')
print(f'Median size  : {statistics.median(sizes):.0f} chars')

# Bucket distribution
buckets = {'<100': 0, '100-500': 0, '500-1000': 0, '1000-2000': 0, '>2000': 0}
for s in sizes:
    if   s < 100:   buckets['<100'] += 1
    elif s < 500:   buckets['100-500'] += 1
    elif s < 1000:  buckets['500-1000'] += 1
    elif s < 2000:  buckets['1000-2000'] += 1
    else:           buckets['>2000'] += 1

print('\nSize distribution:')
for bucket, count in buckets.items():
    bar = '█' * (count // 5)
    print(f'  {bucket:<12} {count:>4}  {bar}')

## ✅ Step 1 Complete!

You now have:
- All repo files loaded as `RawFile` objects
- Python files split by AST (function/class boundaries)
- Rich metadata on every chunk (file, line numbers, node type)

**Next → Step 2:** Embed chunks and store in ChromaDB (`ingestion/embedder.py`)